# 노트북 14. 챗봇 평가 지표 및 방법

> Phase 5 — 품질 관리: 만든 챗봇을 평가하는 법

"잘 되는 것 같다"는 평가가 아닙니다.
체계적이고 반복 가능한 평가 체계가 있어야 개선할 수 있습니다.

**학습 목표**
- LLM 평가가 어려운 이유를 이해한다
- 정확성, 관련성, 충실도, 유해성, 톤 등 평가 기준을 설계할 수 있다
- LLM-as-Judge 패턴으로 자동 평가를 구현할 수 있다
- Pairwise 비교와 RAG Triad 평가를 수행할 수 있다
- LangSmith로 체계적인 평가 파이프라인을 구성할 수 있다

**구성**
| Part | 내용 | 형태 |
|------|------|------|
| Part 1 — 이론 | 평가 기준 + LLM-as-Judge + RAG Triad | 읽고 실행 |
| Part 2 — 실습 | Judge 구현 + Pairwise + LangSmith | 직접 작성 |
| Part 3 — 챌린지 | 나만의 평가 파이프라인 설계 | 강사와 함께 |

In [ ]:
# 환경 설정 — 이 셀을 먼저 실행하세요
!pip install -q google-genai langchain-google-genai langsmith

import os
import json
from google import genai

# API 키 설정 (Colab 환경 기준)
from google.colab import userdata
GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

client = genai.Client(api_key=GEMINI_API_KEY)
MODEL = "gemini-2.5-flash"

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatGoogleGenerativeAI(
    model=MODEL,
    google_api_key=GEMINI_API_KEY,
)

# LangSmith 설정 (선택 — API 키가 있는 경우)
try:
    os.environ["LANGSMITH_API_KEY"] = userdata.get("LANGSMITH_API_KEY")
    os.environ["LANGSMITH_TRACING"] = "true"
    os.environ["LANGSMITH_PROJECT"] = "notebook-14-evaluation"
    print("LangSmith 설정 완료")
except Exception:
    print("LangSmith API 키 없음 — LangSmith 관련 실습은 건너뛰세요")

print("환경 설정 완료")

---

# Part 1 — 이론

개념을 마크다운 설명과 실행 가능한 코드 예시로 배웁니다.
읽고 실행하면서 따라와 주세요.

## 1.1 왜 LLM 평가가 어려운가?

전통 소프트웨어는 `assert result == expected`로 테스트합니다.
하지만 LLM은 다릅니다:

| 전통 소프트웨어 | LLM |
|----------------|-----|
| 정답이 하나 | 좋은 답변이 여러 개 |
| 결정적 출력 | 확률적 출력 (같은 입력, 다른 결과) |
| 단위 테스트 가능 | 주관적 품질 판단 필요 |
| 통과/실패 | 스펙트럼 (얼마나 좋은가?) |

### 평가의 3가지 접근

| 접근 | 방법 | 장점 | 단점 |
|------|------|------|------|
| 인간 평가 | 사람이 직접 채점 | 가장 신뢰성 높음 | 비용 높음, 일관성 낮음 |
| 규칙 기반 | 키워드, 길이, 형식 체크 | 빠르고 저렴 | 품질 판단 불가 |
| **LLM-as-Judge** | LLM이 평가 | 자동화, 확장 가능 | Judge 편향 가능 |

> 실무에서는 세 가지를 조합합니다.
> 규칙 기반으로 기본 필터링 → LLM-as-Judge로 대량 평가 → 인간이 샘플 검수.

## 1.2 평가 기준 설계

"좋은 답변"을 판단하려면 먼저 **무엇이 좋은 건지** 정의해야 합니다.

| 기준 | 질문 | 예시 |
|------|------|------|
| **정확성** (Correctness) | 사실적으로 맞는가? | "서울은 한국의 수도" (O) |
| **관련성** (Relevance) | 질문에 대한 답인가? | Q: 날씨? A: 경제 이야기 (X) |
| **충실도** (Faithfulness) | 주어진 문서에 근거하는가? (RAG) | 문서에 없는 내용을 지어냄 (X) |
| **유해성** (Harmfulness) | 위험/부적절한 내용이 있는가? | 폭력적 표현 (X) |
| **톤** (Tone) | 챗봇 페르소나에 맞는가? | 공식 챗봇인데 반말 (X) |

> 모든 기준을 동시에 평가할 필요는 없습니다.
> 서비스 특성에 맞는 **핵심 기준 2~3개**를 선택하세요.

## 1.3 규칙 기반 평가

가장 간단한 평가부터 시작합니다.
LLM 호출 없이 코드로 체크할 수 있는 기본 품질 지표입니다.

In [ ]:
# 규칙 기반 평가 함수들

def eval_not_empty(response: str) -> dict:
    """응답이 비어있지 않은지 확인합니다."""
    passed = len(response.strip()) > 0
    return {"metric": "not_empty", "score": 1.0 if passed else 0.0}

def eval_length(response: str, min_len: int = 10, max_len: int = 500) -> dict:
    """응답 길이가 적절한 범위 내인지 확인합니다."""
    length = len(response)
    passed = min_len <= length <= max_len
    return {"metric": "length", "score": 1.0 if passed else 0.0, "length": length}

def eval_contains_keywords(response: str, keywords: list[str]) -> dict:
    """응답에 필수 키워드가 포함되어 있는지 확인합니다."""
    found = [kw for kw in keywords if kw in response]
    score = len(found) / len(keywords) if keywords else 0.0
    return {"metric": "keywords", "score": score, "found": found, "missing": [kw for kw in keywords if kw not in response]}

def eval_no_harmful(response: str) -> dict:
    """응답에 위험한 키워드가 없는지 확인합니다."""
    harmful_patterns = ["폭탄", "해킹 방법", "불법", "마약"]
    found = [p for p in harmful_patterns if p in response]
    return {"metric": "no_harmful", "score": 0.0 if found else 1.0, "found": found}

# 테스트
test_response = "서울은 대한민국의 수도이며, 인구는 약 1000만 명입니다."

print(f"응답: '{test_response}'\n")
print(eval_not_empty(test_response))
print(eval_length(test_response))
print(eval_contains_keywords(test_response, ["서울", "수도", "인구"]))
print(eval_no_harmful(test_response))

In [ ]:
# 복합 평가 함수: 여러 규칙을 한번에 적용
def evaluate_response(response: str, keywords: list[str] = None) -> dict:
    """여러 규칙 기반 평가를 한번에 수행합니다."""
    results = {
        "not_empty": eval_not_empty(response),
        "length": eval_length(response),
        "no_harmful": eval_no_harmful(response),
    }

    if keywords:
        results["keywords"] = eval_contains_keywords(response, keywords)

    # 종합 점수: 각 metric의 평균
    scores = [r["score"] for r in results.values()]
    results["overall"] = round(sum(scores) / len(scores), 2)

    return results


# 여러 응답에 대해 한번에 평가
responses = [
    ("서울은 대한민국의 수도이며 인구 약 1000만 명입니다.", ["서울", "수도"]),
    ("", ["서울"]),
    ("파이썬입니다.", ["파이썬", "특징", "장점"]),
]

for resp, kws in responses:
    result = evaluate_response(resp, kws)
    print(f"응답: '{resp[:30]}...' → 종합: {result['overall']}")

> 규칙 기반 평가는 빠르고 저렴하지만, **의미적 품질**(정확성, 관련성)은 판단할 수 없습니다.
> 기본 필터로 사용하고, 의미적 평가는 LLM-as-Judge에 맡깁니다.

## 1.4 LLM-as-Judge 패턴

**LLM-as-Judge**는 LLM에게 평가 기준(루브릭)을 주고,
다른 LLM의 응답을 채점하게 하는 패턴입니다.

```
입력: 질문 + 응답 + 평가 기준
  ↓
Judge LLM: "이 응답은 정확성 4/5, 관련성 5/5..."
```

### Judge 프롬프트 설계 원칙

| 원칙 | 설명 |
|------|------|
| 명확한 기준 | 각 점수가 의미하는 바를 구체적으로 정의 |
| 점수 스케일 | 1 ~ 5 또는 1 ~ 10 (너무 세밀하면 일관성 저하) |
| 근거 요구 | 점수뿐 아니라 이유(reasoning)도 함께 출력 |
| 구조화 출력 | JSON 형식으로 파싱 가능하게 |

### 평가 지표 유형 정리

| 유형 | 방법 | 비용 | 정확도 |
|------|------|------|--------|
| **규칙 기반** | 코드로 체크 (길이, 키워드, 형식) | 무료 | 낮음 |
| **유사도 기반** | 임베딩 유사도, BLEU, ROUGE | 낮음 | 중간 |
| **LLM-as-Judge** | LLM이 채점 | 중간 | 높음 |
| **인간 평가** | 사람이 직접 채점 | 높음 | 가장 높음 |

> 비용-정확도 트레이드오프를 고려하여 단계적으로 적용합니다.
> 규칙 → LLM-as-Judge → 인간 검수 순서가 일반적입니다.

In [ ]:

# 임베딩 기반 유사도 평가 (노트북 12 복습)
# 정답과 AI 응답의 의미적 유사도를 측정

def eval_semantic_similarity(response: str, reference: str) -> dict:
    """임베딩 코사인 유사도로 의미적 일치도를 측정합니다."""
    import numpy as np

    result = client.models.embed_content(
        model="gemini-embedding-001",
        contents=[response, reference],
    )

    vec_resp = np.array(result.embeddings[0].values)
    vec_ref = np.array(result.embeddings[1].values)

    sim = float(
        np.dot(vec_resp, vec_ref)
        / (np.linalg.norm(vec_resp) * np.linalg.norm(vec_ref))
    )

    return {"metric": "semantic_similarity", "score": round(sim, 4)}


# 테스트
pairs = [
    (
        "리스트는 수정 가능하고 튜플은 불가능합니다",
        "리스트는 mutable, 튜플은 immutable입니다",
    ),
    (
        "리스트는 수정 가능하고 튜플은 불가능합니다",
        "오늘 날씨가 좋습니다",
    ),
]

for resp, ref in pairs:
    result = eval_semantic_similarity(resp, ref)
    print(f"응답: '{resp[:30]}...' vs 참조: '{ref[:30]}...'")
    print(f"  유사도: {result['score']}\n")


In [ ]:
# LLM-as-Judge 구현

judge_prompt = ChatPromptTemplate.from_template(
    """당신은 AI 응답 품질을 평가하는 전문 평가자입니다.

아래 질문에 대한 AI의 응답을 평가하세요.

[질문]
{question}

[AI 응답]
{response}

[평가 기준]
각 항목을 1~5점으로 채점하세요.
- 정확성: 사실적으로 올바른가? (1=완전히 틀림, 5=완벽히 정확)
- 관련성: 질문에 대한 답변인가? (1=완전히 무관, 5=정확히 질문에 답함)
- 완성도: 충분히 상세한가? (1=너무 짧음, 5=적절히 상세)

아래 JSON 형식으로 답하세요:
{{
  "accuracy": {{"score": N, "reason": "이유"}},
  "relevance": {{"score": N, "reason": "이유"}},
  "completeness": {{"score": N, "reason": "이유"}},
  "overall": N
}}"""
)

judge_chain = judge_prompt | llm | StrOutputParser()

print("Judge 체인 구성 완료")

### 평가 방법 선택 가이드

```
질문: 정답이 있는가?
  ├─ Yes → 참조 기반 평가
  │    ├─ 정확한 일치 필요? → 규칙 기반 (키워드, 형식)
  │    └─ 의미적 일치 필요? → 임베딩 유사도 or LLM-as-Judge
  └─ No → 참조 없는 평가
       ├─ 기본 품질? → 규칙 기반 (길이, 유해성)
       └─ 의미적 품질? → LLM-as-Judge
```

In [ ]:
# Judge 실행 — 좋은 응답 평가
good_result = judge_chain.invoke({
    "question": "파이썬에서 리스트와 튜플의 차이점은 무엇인가요?",
    "response": "리스트는 대괄호[]로 생성하며 수정 가능(mutable)합니다. 튜플은 소괄호()로 생성하며 수정 불가능(immutable)합니다. 리스트는 요소 추가/삭제가 자유롭고, 튜플은 생성 후 변경할 수 없어 데이터 무결성이 보장됩니다.",
})

print("=== 좋은 응답 평가 ===")
print(good_result)

In [ ]:
# Judge 실행 — 나쁜 응답 평가
bad_result = judge_chain.invoke({
    "question": "파이썬에서 리스트와 튜플의 차이점은 무엇인가요?",
    "response": "파이썬은 인기 있는 프로그래밍 언어입니다.",
})

print("=== 나쁜 응답 평가 ===")
print(bad_result)

### Judge 모델 선택

| 전략 | 설명 | 장점 | 단점 |
|------|------|------|------|
| 동일 모델 | 생성 모델과 같은 모델로 평가 | 비용 절감 | 자기 선호 편향 |
| 상위 모델 | 더 강력한 모델로 평가 | 높은 정확도 | 비용 증가 |
| 다중 Judge | 여러 모델로 평가 후 합산 | 편향 감소 | 비용 높음 |

> **권장**: 생성 모델보다 **같거나 상위 모델**을 Judge로 사용합니다.
> 예: Flash로 생성, Flash 또는 Pro로 평가.

### 루브릭 설계 팁

좋은 Judge 프롬프트는 **구체적인 루브릭**을 가지고 있습니다.

나쁜 예:
```
"정확성을 1~5점으로 평가하세요"
```

좋은 예:
```
"정확성을 1~5점으로 평가하세요.
5: 모든 사실이 정확하고 오해의 소지가 없음
4: 대부분 정확하지만 사소한 부정확함이 있음
3: 핵심은 맞지만 일부 틀린 정보 포함
2: 틀린 정보가 많지만 일부 맞는 부분 있음
1: 완전히 틀리거나 무관한 답변"
```

> 각 점수가 **무엇을 의미하는지** 구체적으로 정의해야 Judge의 일관성이 높아집니다.

> 좋은 응답은 높은 점수, 나쁜 응답(질문과 무관)은 낮은 점수를 받습니다.
> Judge가 **이유(reason)**를 함께 제공하므로, 왜 그런 점수인지 파악할 수 있습니다.

### Judge 결과 파싱

Judge의 JSON 출력을 파싱하여 점수를 추출하는 유틸리티를 만듭니다.

In [ ]:
# Judge 결과 파싱 유틸리티
def parse_judge_result(raw: str) -> dict:
    """Judge의 JSON 응답을 파싱합니다. 코드 블록 감싸기를 처리합니다."""
    clean = raw.strip()
    if clean.startswith("```"):
        clean = clean.split("\n", 1)[1]
    if clean.endswith("```"):
        clean = clean.rsplit("```", 1)[0]
    try:
        return json.loads(clean)
    except json.JSONDecodeError:
        return {"error": "JSON 파싱 실패", "raw": raw}

# 파싱 테스트
parsed = parse_judge_result(good_result)
if "error" not in parsed:
    print(f"정확성: {parsed['accuracy']['score']}/5 — {parsed['accuracy']['reason']}")
    print(f"관련성: {parsed['relevance']['score']}/5 — {parsed['relevance']['reason']}")
    print(f"완성도: {parsed['completeness']['score']}/5 — {parsed['completeness']['reason']}")
    print(f"종합:   {parsed['overall']}/5")
else:
    print(parsed)

In [ ]:
# Structured Output을 활용한 Judge (노트북 8 복습)
# JSON 파싱 오류를 줄이기 위해 with_structured_output 사용

from pydantic import BaseModel, Field


class JudgeResult(BaseModel):
    accuracy: int = Field(description="정확성 점수 (1~5)")
    relevance: int = Field(description="관련성 점수 (1~5)")
    completeness: int = Field(description="완성도 점수 (1~5)")
    overall: float = Field(description="종합 점수 (1~5)")
    reasoning: str = Field(description="평가 근거")


structured_judge = llm.with_structured_output(JudgeResult)

judge_msg = f"""아래 질문에 대한 AI 응답을 평가하세요.

[질문] 파이썬에서 예외 처리 방법은?

[AI 응답] try-except 블록을 사용합니다. try에 실행할 코드를,
except에 예외 처리 코드를 작성합니다.
finally로 항상 실행되는 코드를 추가할 수 있습니다."""

result = structured_judge.invoke(judge_msg)

print(f"정확성: {result.accuracy}/5")
print(f"관련성: {result.relevance}/5")
print(f"완성도: {result.completeness}/5")
print(f"종합:   {result.overall}/5")
print(f"근거:   {result.reasoning}")
print(f"\n타입: {type(result).__name__} (Pydantic 모델 → JSON 파싱 오류 없음)")


In [ ]:
# 자기 일관성 검증: 같은 평가를 3회 반복하여 일관성 확인

question = "파이썬에서 리스트와 튜플의 차이점은 무엇인가요?"
response = "리스트는 수정 가능하고 튜플은 수정 불가능합니다."

scores = []
for trial in range(3):
    raw = judge_chain.invoke({"question": question, "response": response})
    parsed = parse_judge_result(raw)
    if "error" not in parsed:
        scores.append(parsed.get("overall", 0))

if scores:
    avg = sum(scores) / len(scores)
    spread = max(scores) - min(scores)
    print(f"3회 반복 평가: {scores}")
    print(f"평균: {avg:.1f}, 편차: {spread}")
    print(f"일관성: {'높음' if spread <= 1 else '낮음 (편차 > 1)'}")

## 1.5 배치 평가

여러 질문-응답 쌍을 한번에 평가하는 배치 평가를 구현합니다.
이것이 체계적 평가의 기본입니다.

In [ ]:
# 평가 데이터셋 준비
eval_dataset = [
    {
        "question": "HTML에서 하이퍼링크를 만드는 태그는?",
        "response": "<a> 태그를 사용합니다. href 속성에 URL을 지정합니다. 예: <a href='https://example.com'>링크</a>",
    },
    {
        "question": "HTTP GET과 POST의 차이점은?",
        "response": "GET은 데이터 조회에 사용되며 URL에 파라미터가 노출됩니다. POST는 데이터 전송에 사용되며 요청 본문에 데이터를 담습니다.",
    },
    {
        "question": "JavaScript에서 == 와 === 의 차이는?",
        "response": "둘 다 비교 연산자입니다.",  # 불완전한 답변
    },
    {
        "question": "CSS에서 margin과 padding의 차이는?",
        "response": "오늘 날씨가 좋습니다.",  # 완전히 무관한 답변
    },
]

# 배치 평가 실행
print("=== 배치 평가 결과 ===\n")
scores = []

for i, item in enumerate(eval_dataset):
    raw = judge_chain.invoke(item)
    parsed = parse_judge_result(raw)

    if "error" not in parsed:
        overall = parsed.get("overall", 0)
        scores.append(overall)
        print(f"[{i}] Q: {item['question'][:40]}...")
        print(f"    점수: {overall}/5")
    else:
        print(f"[{i}] 파싱 실패")
    print()

if scores:
    print(f"평균 점수: {sum(scores) / len(scores):.1f}/5")
    print(f"최고: {max(scores)}/5, 최저: {min(scores)}/5")

## 1.6 Pairwise 비교

**Pairwise 비교**는 두 응답을 나란히 놓고 "어느 것이 더 나은가"를 판정합니다.
A/B 테스트나 모델 교체 시 유용합니다.

```
질문: "파이썬이란?"
응답 A: "프로그래밍 언어입니다."
응답 B: "파이썬은 간결한 문법으로 유명한 범용 프로그래밍 언어입니다."
Judge: "응답 B가 더 상세하고 유용합니다. 승자: B"
```

In [ ]:
# Pairwise 비교 구현
pairwise_prompt = ChatPromptTemplate.from_template(
    """아래 질문에 대한 두 가지 AI 응답을 비교하세요.

[질문]
{question}

[응답 A]
{response_a}

[응답 B]
{response_b}

어느 응답이 더 나은지 판단하세요.
아래 JSON 형식으로 답하세요:
{{
  "winner": "A" 또는 "B" 또는 "TIE",
  "reason": "판단 근거를 2~3문장으로 설명",
  "score_a": N,
  "score_b": N
}}"""
)

pairwise_chain = pairwise_prompt | llm | StrOutputParser()

# 비교 실행
result = pairwise_chain.invoke({
    "question": "머신러닝과 딥러닝의 차이점은?",
    "response_a": "머신러닝은 데이터에서 패턴을 학습하는 기술이고, 딥러닝은 그 중 신경망을 사용하는 방법입니다.",
    "response_b": "머신러닝은 AI의 하위 분야로, 데이터를 통해 모델을 학습시키는 기법입니다. 딥러닝은 머신러닝의 하위 분야로, 여러 층의 인공 신경망을 사용합니다. 머신러닝은 특성 추출이 필요하지만, 딥러닝은 원시 데이터에서 직접 학습할 수 있습니다.",
})

print("=== Pairwise 비교 결과 ===")
print(result)

### 평가 데이터셋 설계 원칙

| 원칙 | 설명 |
|------|------|
| **다양성** | 쉬운/어려운, 짧은/긴, 다양한 주제 포함 |
| **경계 케이스** | 모호한 질문, 문서에 없는 질문, 다국어 |
| **균형** | 좋은 응답과 나쁜 응답이 나올 수 있는 질문 혼합 |
| **현실성** | 실제 사용자가 할 법한 질문 |
| **크기** | 최소 20~50개, 이상적으로 100개 이상 |

> 처음부터 완벽한 데이터셋을 만들 필요는 없습니다.
> **핵심 시나리오 10개로 시작**하고, 실제 사용 로그를 보며 점진적으로 확장하세요.

> **위치 편향(Position Bias) 주의**: Judge LLM은 A를 먼저 보기 때문에
> A를 선호하는 경향이 있을 수 있습니다.
> 이를 방지하려면 A/B 순서를 바꿔서 두 번 평가하고, 결과가 일관적인지 확인합니다.

### 참조 기반 vs 참조 없는 평가

| 유형 | 필요한 것 | 장점 | 단점 |
|------|----------|------|------|
| **참조 기반** | 질문 + 정답 + AI 응답 | 정확한 비교 가능 | 정답 작성 비용 |
| **참조 없는** | 질문 + AI 응답만 | 정답 불필요 | 정확도 판단 제한 |

참조 기반은 정확성 평가에, 참조 없는 평가는 유창성/톤 평가에 적합합니다.

```python
# 참조 기반 Judge 프롬프트
"""
[정답] {reference}
[AI 응답] {response}

정답과 비교하여 AI 응답의 정확성을 평가하세요.
"""
```

In [ ]:
# 위치 편향 검증: A/B 순서를 바꿔서 재평가
result_swapped = pairwise_chain.invoke({
    "question": "머신러닝과 딥러닝의 차이점은?",
    "response_a": "머신러닝은 AI의 하위 분야로, 데이터를 통해 모델을 학습시키는 기법입니다. 딥러닝은 머신러닝의 하위 분야로, 여러 층의 인공 신경망을 사용합니다. 머신러닝은 특성 추출이 필요하지만, 딥러닝은 원시 데이터에서 직접 학습할 수 있습니다.",
    "response_b": "머신러닝은 데이터에서 패턴을 학습하는 기술이고, 딥러닝은 그 중 신경망을 사용하는 방법입니다.",
})

print("=== 순서 교체 후 결과 ===")
print(result_swapped)
print("\n순서를 바꿔도 같은 응답이 승자로 선택되면 일관된 평가입니다")

In [ ]:
# 참조 기반 평가: 정답과 비교
ref_judge_prompt = ChatPromptTemplate.from_template(
    """아래 정답을 기준으로 AI 응답의 정확성을 1~5점으로 평가하세요.

[질문] {question}
[정답] {reference}
[AI 응답] {response}

JSON으로 답하세요: {{"score": N, "reason": "이유"}}"""
)

ref_judge = ref_judge_prompt | llm | StrOutputParser()

# 테스트
test = {
    "question": "HTTP 상태코드 200의 의미는?",
    "reference": "요청이 성공적으로 처리되었음을 나타냅니다 (OK).",
    "response": "200은 OK를 의미하며, 서버가 요청을 성공적으로 처리했다는 뜻입니다.",
}

result = ref_judge.invoke(test)

print(f"Q: {test['question']}")
print(f"정답: {test['reference']}")
print(f"AI: {test['response']}")
print(f"평가: {result}")

## 1.7 RAG Triad: RAG 특화 평가

노트북 13에서 미리 소개한 RAG 평가의 3요소를 구현합니다.

| 지표 | 평가 대상 | 질문 |
|------|----------|------|
| **Context Relevance** | 검색된 문서 | 검색된 문서가 질문과 관련 있는가? |
| **Answer Faithfulness** | 답변 vs 문서 | 답변이 검색된 문서에 근거하는가? |
| **Answer Relevance** | 답변 vs 질문 | 답변이 질문에 대한 답인가? |

```
질문 ←── Answer Relevance ──→ 답변
  │                              ↑
  │                    Answer Faithfulness
  │                              │
Context Relevance ──→ 검색 문서 ──┘
```

In [ ]:
# RAG Triad 평가 구현

# 1. Context Relevance
context_relevance_prompt = ChatPromptTemplate.from_template(
    """검색된 문서가 질문과 관련이 있는지 1~5점으로 평가하세요.

[질문] {question}
[검색 문서] {context}

JSON으로 답하세요: {{"score": N, "reason": "이유"}}"""
)

# 2. Answer Faithfulness
faithfulness_prompt = ChatPromptTemplate.from_template(
    """답변이 아래 문서에만 근거하는지 1~5점으로 평가하세요.
문서에 없는 내용을 포함하면 감점합니다.

[문서] {context}
[답변] {answer}

JSON으로 답하세요: {{"score": N, "reason": "이유"}}"""
)

# 3. Answer Relevance
answer_relevance_prompt = ChatPromptTemplate.from_template(
    """답변이 질문에 대한 적절한 답인지 1~5점으로 평가하세요.

[질문] {question}
[답변] {answer}

JSON으로 답하세요: {{"score": N, "reason": "이유"}}"""
)

cr_chain = context_relevance_prompt | llm | StrOutputParser()
faith_chain = faithfulness_prompt | llm | StrOutputParser()
ar_chain = answer_relevance_prompt | llm | StrOutputParser()

print("RAG Triad 평가 체인 3개 구성 완료")

In [ ]:
# 길이 편향 검증: 같은 내용, 다른 길이
length_test = pairwise_chain.invoke({
    "question": "변수란 무엇인가요?",
    "response_a": "값을 저장하는 이름입니다.",
    "response_b": "변수(Variable)란 프로그래밍에서 데이터를 저장하기 위한 "
                  "메모리 공간에 붙이는 이름입니다. 변수를 통해 데이터를 "
                  "저장하고, 읽고, 수정할 수 있습니다. 변수 이름은 알파벳, "
                  "숫자, 언더스코어로 구성되며 숫자로 시작할 수 없습니다.",
})

print("=== 길이 편향 테스트 ===")
print(f"응답 A: 짧은 답변 (10자)")
print(f"응답 B: 긴 답변 (100자)")
print(f"결과: {length_test}")
print("\n같은 정확도라면 긴 답변이 선호되는 경향이 있습니다 (길이 편향)")


In [ ]:
# RAG Triad 평가 실행
question = "파이썬에서 리스트 정렬 방법은?"
context = "파이썬 리스트는 sort() 메서드로 제자리 정렬하거나, sorted() 함수로 새 리스트를 반환할 수 있습니다. reverse=True로 내림차순, key 파라미터로 정렬 기준을 지정합니다."
answer = "파이썬에서 리스트를 정렬하려면 sort() 메서드나 sorted() 함수를 사용합니다. sort()는 원본을 수정하고, sorted()는 새 리스트를 만듭니다. reverse=True로 내림차순 정렬이 가능합니다."

# 3가지 평가 실행
cr = cr_chain.invoke({"question": question, "context": context})
faith = faith_chain.invoke({"context": context, "answer": answer})
ar = ar_chain.invoke({"question": question, "answer": answer})

print("=== RAG Triad 평가 ===")
print(f"Q: {question}\n")
print(f"Context Relevance: {cr}")
print(f"Answer Faithfulness: {faith}")
print(f"Answer Relevance: {ar}")

## 1.8 LangSmith Evaluation

**LangSmith**는 LangChain 생태계의 평가 플랫폼입니다.
테스트 케이스(Dataset)를 등록하고, Evaluator를 정의하여 자동 평가를 실행합니다.

```
1. Dataset 생성: 질문 + 기대 답변 등록
2. Target 정의: 평가할 체인/함수
3. Evaluator 정의: 점수 계산 로직
4. evaluate() 실행: 자동 평가 + 결과 대시보드
```

> 아래 코드는 LangSmith API 키가 있어야 실행됩니다.
> 키가 없으면 코드를 읽기만 하고, 개념을 이해하세요.

In [ ]:
# LangSmith Dataset 생성
from langsmith import Client as LangSmithClient

ls_client = LangSmithClient()

# 데이터셋 생성 (이미 존재하면 재사용)
dataset_name = "notebook-14-demo"

try:
    dataset = ls_client.create_dataset(dataset_name)
    print(f"데이터셋 '{dataset_name}' 생성 완료")
except Exception:
    dataset = ls_client.read_dataset(dataset_name=dataset_name)
    print(f"데이터셋 '{dataset_name}' 이미 존재 — 재사용")

# 테스트 케이스 등록
examples = [
    {
        "inputs": {"question": "파이썬의 GIL이란?"},
        "outputs": {"answer": "Global Interpreter Lock의 약자로, 한 번에 하나의 스레드만 파이썬 바이트코드를 실행할 수 있게 하는 잠금 메커니즘입니다."},
    },
    {
        "inputs": {"question": "REST API의 특징은?"},
        "outputs": {"answer": "무상태성(Stateless), 클라이언트-서버 구조, 캐시 가능, 계층화 시스템, 통일된 인터페이스 등이 있습니다."},
    },
    {
        "inputs": {"question": "Git에서 rebase와 merge의 차이는?"},
        "outputs": {"answer": "merge는 두 브랜치의 이력을 그대로 유지하며 병합하고, rebase는 커밋 이력을 재정렬하여 선형적으로 만듭니다."},
    },
]

ls_client.create_examples(
    inputs=[e["inputs"] for e in examples],
    outputs=[e["outputs"] for e in examples],
    dataset_id=dataset.id,
)

print(f"테스트 케이스 {len(examples)}개 등록 완료")

### 평가 비용 추정

| 평가 유형 | 추가 API 호출 | 비용 영향 |
|-----------|--------------|----------|
| 규칙 기반 | 0회 | 무료 |
| LLM-as-Judge (1기준) | 1회/응답 | 생성 비용의 약 50~100% |
| LLM-as-Judge (3기준) | 1회/응답 (복합 프롬프트) | 생성 비용과 유사 |
| Pairwise | 1~2회/쌍 | 생성 비용의 2배 |
| RAG Triad | 3회/응답 | 생성 비용의 3배 |

> 100개 테스트 케이스를 3가지 기준으로 평가하면, 약 100회의 추가 API 호출입니다.
> Judge에 저렴한 모델(Flash)을 사용하면 비용을 절감할 수 있습니다.

In [ ]:
# LangSmith Evaluator 정의 + 평가 실행
from langsmith.evaluation import evaluate, EvaluationResult

# 평가할 함수 (target)
def my_chatbot(inputs: dict) -> dict:
    """평가 대상 체인 — 질문을 받아 답변을 반환합니다."""
    response = llm.invoke(inputs["question"])
    content = response.content if isinstance(response.content, str) else str(response.content)
    return {"answer": content}

# Evaluator 1: 응답 비어있지 않은지
def not_empty_evaluator(run, example) -> EvaluationResult:
    output = run.outputs.get("answer", "")
    return EvaluationResult(key="not_empty", score=1.0 if len(output.strip()) > 0 else 0.0)

# Evaluator 2: 키워드 포함 여부
def keyword_evaluator(run, example) -> EvaluationResult:
    output = run.outputs.get("answer", "")
    reference = example.outputs.get("answer", "")
    # 참조 답변에서 핵심 단어 추출 (간단한 방법: 3자 이상 단어)
    ref_words = [w for w in reference.split() if len(w) >= 3]
    if not ref_words:
        return EvaluationResult(key="keyword_match", score=0.0)
    found = sum(1 for w in ref_words if w in output)
    return EvaluationResult(key="keyword_match", score=found / len(ref_words))

# 평가 실행
results = evaluate(
    my_chatbot,
    data=dataset_name,
    evaluators=[not_empty_evaluator, keyword_evaluator],
    experiment_prefix="demo-eval",
)

print("평가 완료 — LangSmith 대시보드에서 결과를 확인하세요")

> **LangSmith 대시보드**에서 확인할 수 있는 것:
> - 각 테스트 케이스별 점수
> - Evaluator별 평균 점수
> - 실험 간 비교 (모델 A vs B)
> - 트레이스 (각 호출의 상세 로그)

## 1.9 평가 파이프라인 전체 구조

체계적인 평가 파이프라인의 전체 흐름:

```
1. 평가 데이터셋 구축
   └─ 질문 + 기대 답변 + 참조 문서(RAG용)

2. 평가 기준 정의
   ├─ 규칙 기반: 길이, 형식, 키워드
   ├─ LLM-as-Judge: 정확성, 관련성, 완성도
   └─ RAG 특화: Context Relevance, Faithfulness

3. 자동 평가 실행
   └─ LangSmith evaluate() 또는 자체 스크립트

4. 결과 분석
   ├─ 평균/분포 확인
   ├─ 낮은 점수 케이스 검토
   └─ 인간 검수 (샘플)

5. 개선 → 재평가 (반복)
```

---

### 생각해보기

1. LLM-as-Judge에서 Judge 모델 자체가 편향되어 있다면 어떻게 감지할 수 있을까요? Judge의 평가가 인간 평가와 얼마나 일치하는지 측정하는 방법은?
2. Pairwise 비교에서 위치 편향 외에 다른 편향(예: 길이 편향 — 긴 답변 선호)이 있을 수 있을까요? 이를 방지하는 방법은?
3. RAG Triad의 세 지표 중 하나만 개선할 수 있다면, 어떤 지표를 선택하겠습니까? 그 이유는?

---

# Part 2 — 실습

이론에서 배운 개념을 직접 코드로 작성해봅니다.
TODO 부분을 채워주세요. 막히면 정답 주석을 확인하세요.

### 오류 분석 패턴

평가 결과에서 낮은 점수를 받은 케이스를 분석하는 것이 개선의 핵심입니다.

```
1. 낮은 점수 케이스 수집 (overall <= 2)
2. 오류 유형 분류:
   - 사실 오류: 잘못된 정보 (→ RAG 검색 개선)
   - 관련성 부족: 엉뚱한 답변 (→ 프롬프트 개선)
   - 불완전: 핵심 누락 (→ 검색 k값 증가)
   - 할루시네이션: 지어낸 정보 (→ Faithfulness 검증 추가)
3. 유형별 개선 방안 도출
4. 개선 적용 → 재평가
```

> 오류를 **분류**하는 것이 중요합니다.
> "점수가 낮다"만으로는 어디를 고쳐야 하는지 알 수 없습니다.

In [ ]:
# RAG Triad 통합 평가 함수
def evaluate_rag_triad(question: str, context: str, answer: str) -> dict:
    """RAG Triad 3가지 지표를 한번에 평가합니다."""
    results = {}

    # Context Relevance
    cr_raw = cr_chain.invoke({"question": question, "context": context})
    cr = parse_judge_result(cr_raw)
    results["context_relevance"] = cr.get("score", 0) if "error" not in cr else 0

    # Answer Faithfulness
    faith_raw = faith_chain.invoke({"context": context, "answer": answer})
    faith = parse_judge_result(faith_raw)
    results["faithfulness"] = faith.get("score", 0) if "error" not in faith else 0

    # Answer Relevance
    ar_raw = ar_chain.invoke({"question": question, "answer": answer})
    ar = parse_judge_result(ar_raw)
    results["answer_relevance"] = ar.get("score", 0) if "error" not in ar else 0

    # 종합 점수
    scores = list(results.values())
    results["overall"] = round(sum(scores) / len(scores), 1) if scores else 0

    return results


# 테스트
triad = evaluate_rag_triad(
    question="파이썬 리스트 정렬 방법은?",
    context="파이썬 리스트는 sort() 메서드나 sorted() 함수로 정렬합니다.",
    answer="sort() 메서드나 sorted() 함수를 사용합니다.",
)

print("=== RAG Triad 종합 ===")
for k, v in triad.items():
    print(f"  {k}: {v}")

### 실습 14-1: LLM-as-Judge 프롬프트 설계 + 채점

고객 상담 챗봇의 응답을 평가하는 Judge를 설계하세요.

**요구사항**
1. 평가 기준: 정확성, 공감도(empathy), 해결 제시 여부
2. 각 기준 1~5점, 이유 포함, JSON 출력
3. 아래 3개 테스트 케이스를 평가
4. 결과를 파싱하여 점수 요약 출력

In [ ]:
test_cases = [
    {
        "question": "주문한 상품이 3일째 안 오는데 어떻게 된 건가요?",
        "response": "불편을 드려 정말 죄송합니다. 주문번호를 알려주시면 배송 상태를 바로 확인해드리겠습니다. 만약 배송 지연이 확인되면 우선 재배송 또는 환불 처리를 도와드리겠습니다.",
    },
    {
        "question": "결제했는데 포인트가 적립이 안 됐어요.",
        "response": "포인트는 결제 후 24시간 이내에 자동 적립됩니다.",
    },
    {
        "question": "환불 요청한 지 일주일인데 아직 처리가 안 됐습니다.",
        "response": "그건 저희 부서 담당이 아닙니다.",
    },
]

# TODO: Judge 프롬프트 설계 + 채점 + 결과 파싱

# 정답
# cs_judge_prompt = ChatPromptTemplate.from_template(
#     """당신은 고객 상담 응답 품질 평가 전문가입니다.

# [고객 질문] {question}
# [상담 응답] {response}

# 평가 기준 (각 1~5점):
# - 정확성: 정보가 정확하고 오해의 소지가 없는가?
# - 공감도: 고객의 불만/감정에 공감을 표현했는가?
# - 해결제시: 구체적인 해결 방법이나 다음 단계를 제시했는가?

# JSON 형식: {{"accuracy": {{"score": N, "reason": "..."}}, "empathy": {{"score": N, "reason": "..."}}, "solution": {{"score": N, "reason": "..."}}, "overall": N}}"""
# )

# cs_judge = cs_judge_prompt | llm | StrOutputParser()

# for i, case in enumerate(test_cases):
#     raw = cs_judge.invoke(case)
#     parsed = parse_judge_result(raw)
#     if "error" not in parsed:
#         print(f"[{i}] Q: {case['question'][:40]}...")
#         print(f"    정확성: {parsed['accuracy']['score']}/5")
#         print(f"    공감도: {parsed['empathy']['score']}/5")
#         print(f"    해결제시: {parsed['solution']['score']}/5")
#         print(f"    종합: {parsed['overall']}/5\n")
#     else:
#         print(f"[{i}] 파싱 실패: {parsed}\n")

print("TODO를 완성해주세요")

In [ ]:

# # 토큰 효율성 평가
# # 답변이 지나치게 길거나 짧지 않은지 확인

# def eval_token_efficiency(question: str, response: str) -> dict:
#     """답변의 토큰 효율성을 평가합니다. 질문 대비 적절한 길이인지 확인."""
#     q_len = len(question)
#     r_len = len(response)
#     ratio = r_len / q_len if q_len > 0 else 0

#     # 적절한 비율: 질문의 2~10배
#     if 2 <= ratio <= 10:
#         score = 1.0
#         comment = "적절한 길이"
#     elif ratio < 1:
#         score = 0.3
#         comment = "너무 짧음 (질문보다 짧은 답변)"
#     elif ratio > 20:
#         score = 0.5
#         comment = "과도하게 김 (불필요한 내용 포함 가능)"
#     else:
#         score = 0.7
#         comment = "약간 짧거나 김"

#     return {
#         "metric": "token_efficiency",
#         "score": score,
#         "ratio": round(ratio, 1),
#         "comment": comment,
#     }


# # 테스트
# tests = [
#     ("파이썬이란?", "프로그래밍 언어입니다."),
#     (
#         "파이썬이란?",
#         "파이썬은 1991년 귀도 반 로썸이 만든 범용 프로그래밍 언어입니다. "
#         "간결한 문법이 특징이며, 웹 개발, 데이터 분석, AI 등 다양한 분야에서 사용됩니다.",
#     ),
# ]

# for q, r in tests:
#     result = eval_token_efficiency(q, r)
#     print(f"Q: {q} → A: {r[:40]}...")
#     print(f"  비율: {result['ratio']}x, 점수: {result['score']}, {result['comment']}\n")


In [ ]:

# # 할루시네이션 사례: Faithfulness가 낮은 경우
# triad_bad = evaluate_rag_triad(
#     question="파이썬 리스트 정렬 방법은?",
#     context="파이썬 리스트는 sort() 메서드나 sorted() 함수로 정렬합니다.",
#     answer="파이썬에서는 numpy.sort(), pandas.sort_values(), "
#            "heapq.nlargest() 등 다양한 정렬 방법이 있습니다.",
# )

# print("=== 할루시네이션 사례 ===")
# for k, v in triad_bad.items():
#     print(f"  {k}: {v}")

# print("\n답변이 문서에 없는 numpy, pandas를 언급하여 Faithfulness가 낮습니다")


### 실습 14-2: Pairwise 비교 구현

두 가지 응답 스타일을 비교하는 Pairwise 평가를 구현하세요.

**요구사항**
1. 동일 질문에 대해 "간결한 응답" vs "상세한 응답" 비교
2. 위치 편향 검증: A/B 순서를 바꿔서 재평가
3. 2개 질문으로 테스트
4. 일관성 여부를 출력

In [ ]:
comparison_data = [
    {
        "question": "클라우드 컴퓨팅이란?",
        "short": "인터넷을 통해 컴퓨팅 자원을 제공하는 기술입니다.",
        "detailed": "클라우드 컴퓨팅은 인터넷을 통해 서버, 스토리지, 데이터베이스, 네트워킹 등의 컴퓨팅 자원을 온디맨드로 제공하는 기술입니다. IaaS, PaaS, SaaS 세 가지 서비스 모델이 있으며, AWS, Azure, GCP가 대표적인 제공업체입니다.",
    },
    {
        "question": "API란 무엇인가요?",
        "short": "소프트웨어 간 통신 규약입니다.",
        "detailed": "API(Application Programming Interface)는 소프트웨어 컴포넌트 간 상호작용을 위한 인터페이스입니다. REST API는 HTTP 메서드(GET, POST, PUT, DELETE)를 사용하며, JSON 형식으로 데이터를 주고받습니다. 웹 서비스, 모바일 앱, 마이크로서비스 아키텍처의 핵심 기술입니다.",
    },
]

# TODO: Pairwise 비교 + 위치 편향 검증

# 정답
# for item in comparison_data:
#     # 원래 순서: short=A, detailed=B
#     r1 = pairwise_chain.invoke({
#         "question": item["question"],
#         "response_a": item["short"],
#         "response_b": item["detailed"],
#     })
#     p1 = parse_judge_result(r1)

#     # 순서 교체: detailed=A, short=B
#     r2 = pairwise_chain.invoke({
#         "question": item["question"],
#         "response_a": item["detailed"],
#         "response_b": item["short"],
#     })
#     p2 = parse_judge_result(r2)

#     print(f"Q: {item['question']}")
#     w1 = p1.get('winner', '?') if 'error' not in p1 else '?'
#     w2 = p2.get('winner', '?') if 'error' not in p2 else '?'
#     print(f"  원래 순서 승자: {w1}")
#     print(f"  교체 순서 승자: {w2}")
#     # 일관성: 원래 B 승자 → 교체 시 A 승자면 일관적
#     consistent = (w1 == 'B' and w2 == 'A') or (w1 == 'A' and w2 == 'B') or w1 == 'TIE'
#     print(f"  일관성: {'일관됨' if consistent else '불일치 (위치 편향 의심)'}\n")

# print("TODO를 완성해주세요")

### 실습 14-3: LangSmith Dataset + 자동 평가

LangSmith에 평가 데이터셋을 생성하고, 자동 평가를 실행하세요.

**요구사항**
1. 5개 테스트 케이스로 데이터셋 생성 (dataset_name: "practice-14-3")
2. Evaluator 2개 정의: not_empty + LLM-as-Judge 기반 정확성
3. `evaluate()` 실행
4. (LangSmith 키 없으면) 자체 배치 평가로 대체

In [ ]:
practice_examples = [
    {"question": "Docker의 장점은?", "answer": "환경 일관성, 빠른 배포, 리소스 효율성"},
    {"question": "SQL의 JOIN 종류는?", "answer": "INNER, LEFT, RIGHT, FULL OUTER, CROSS JOIN"},
    {"question": "Git branch 전략 중 하나는?", "answer": "Git Flow: main, develop, feature, release, hotfix 브랜치 사용"},
    {"question": "HTTP 상태코드 404는?", "answer": "Not Found. 요청한 리소스를 서버에서 찾을 수 없음"},
    {"question": "OOP의 4대 원칙은?", "answer": "캡슐화, 상속, 다형성, 추상화"},
]

# TODO: LangSmith Dataset 생성 + 평가 실행 (또는 자체 배치 평가)

# 정답
# # # LangSmith 방식
# try:
#     ds_name = "practice-14-3"
#     try:
#         ds = ls_client.create_dataset(ds_name)
#     except Exception:
#         ds = ls_client.read_dataset(dataset_name=ds_name)

#     ls_client.create_examples(
#         inputs=[{"question": e["question"]} for e in practice_examples],
#         outputs=[{"answer": e["answer"]} for e in practice_examples],
#         dataset_id=ds.id,
#     )

#     def accuracy_evaluator(run, example) -> EvaluationResult:
#         output = run.outputs.get("answer", "")
#         reference = example.outputs.get("answer", "")
#         raw = judge_chain.invoke({
#             "question": example.inputs["question"],
#             "response": output,
#         })
#         parsed = parse_judge_result(raw)
#         score = parsed.get("overall", 0) / 5.0 if "error" not in parsed else 0.0
#         return EvaluationResult(key="accuracy", score=score)

#     results = evaluate(
#         my_chatbot,
#         data=ds_name,
#         evaluators=[not_empty_evaluator, accuracy_evaluator],
#         experiment_prefix="practice-eval",
#     )
#     print("LangSmith 평가 완료")

# except Exception as e:
#     print(f"LangSmith 불가: {e}")
#     print("자체 배치 평가로 대체합니다\n")
#     for ex in practice_examples:
#         resp = llm.invoke(ex["question"]).content
#         raw = judge_chain.invoke({"question": ex["question"], "response": resp})
#         parsed = parse_judge_result(raw)
#         overall = parsed.get("overall", "?") if "error" not in parsed else "?"
#         print(f"Q: {ex['question']}")
#         print(f"  점수: {overall}/5\n")

print("TODO를 완성해주세요")

---

### 생각해보기

1. 실습 14-1에서 세 번째 응답("그건 저희 부서 담당이 아닙니다")이 낮은 점수를 받은 이유는 무엇인가요? 이 응답을 개선한다면 어떻게 바꾸겠습니까?
2. 실습 14-2에서 간결한 응답과 상세한 응답 중 항상 상세한 것이 좋은 건 아닙니다. 간결한 응답이 더 적합한 상황은 언제일까요?
3. LangSmith 없이도 체계적인 평가가 가능할까요? 자체 평가 파이프라인을 구축한다면 어떤 도구와 구조를 사용하겠습니까?

---

# Part 3 — 챌린지

이론과 실습에서 배운 내용을 응용하여 스스로 설계하고 구현합니다.
정답이 제공되지 않으며, 강사와 함께 진행합니다.

### 챌린지 14-1: 나만의 평가 파이프라인 설계 (난이도: 상)

> 이 챌린지는 정답이 제공되지 않습니다. 강사와 함께 진행하세요.

특정 도메인의 챗봇을 위한 종합 평가 파이프라인을 설계하세요.

**조건**
- 도메인 선택: 고객 상담 / IT 헬프데스크 / 교육 튜터 중 하나
- 평가 기준 3개 이상 정의 (도메인에 맞는 기준)
- 테스트 케이스 5개 이상 작성
- 규칙 기반 + LLM-as-Judge 평가를 결합

**추가 과제**
- 두 가지 다른 system prompt로 생성한 응답을 Pairwise 비교
- 평가 결과를 표로 정리하고, 어떤 prompt가 더 나은지 결론 도출

**힌트**
- 도메인에 맞는 평가 기준을 먼저 정의하세요 (예: 헬프데스크 → 기술 정확성, 단계별 안내)
- 좋은 응답과 나쁜 응답을 모두 포함하여 평가자가 구분하는지 확인하세요

In [ ]:
# 여기에 코드를 작성하세요
# === 챌린지 14-1 답안 ===
# 도메인: IT 헬프데스크

# ── 1단계: 평가 기준 정의 ──
# 1. 기술 정확성: 제시한 해결 방법이 기술적으로 올바른가
# 2. 단계별 안내: 사용자가 따라할 수 있는 구체적 절차를 제공하는가
# 3. 공감 표현: 사용자의 불편에 공감하고 친절하게 응대하는가

# ── 2단계: 테스트 케이스 + 두 가지 프롬프트로 응답 생성 ──
from pydantic import BaseModel, Field
from typing import Literal

test_cases = [
    {"question": "노트북 와이파이가 안 됩니다", "topic": "네트워크"},
    {"question": "프린터가 인쇄를 안 해요", "topic": "프린터"},
    {"question": "비밀번호를 잊어버렸어요", "topic": "계정"},
    {"question": "엑셀 파일이 안 열려요", "topic": "소프트웨어"},
    {"question": "화면이 갑자기 파란색이 됐어요", "topic": "하드웨어"},
]

prompt_a = "당신은 IT 헬프데스크 상담원입니다. 기술적으로 정확하고 단계별로 안내하세요. 공감을 표현하세요."
prompt_b = "IT 질문에 답하세요."  # 최소한의 프롬프트

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import SystemMessage, HumanMessage

llm = ChatGoogleGenerativeAI(model=MODEL, google_api_key=GEMINI_API_KEY)

responses = {"A": [], "B": []}
for tc in test_cases:
    for label, sys_prompt in [("A", prompt_a), ("B", prompt_b)]:
        resp = llm.invoke([SystemMessage(sys_prompt), HumanMessage(tc["question"])])
        responses[label].append(resp.content)

# ── 3단계: 규칙 기반 평가 ──
def rule_eval(answer):
    has_steps = any(marker in answer for marker in ["1.", "1)", "먼저", "첫째", "Step"])
    has_empathy = any(kw in answer for kw in ["불편", "죄송", "걱정", "도와", "어려우"])
    length_ok = len(answer) >= 80
    return {"steps": has_steps, "empathy": has_empathy, "length_ok": length_ok}

# ── 4단계: LLM-as-Judge 평가 ──
class HelpDeskScore(BaseModel):
    accuracy: int = Field(description="기술 정확성 (1~5)")
    step_guide: int = Field(description="단계별 안내 구체성 (1~5)")
    empathy: int = Field(description="공감 및 친절도 (1~5)")

judge = llm.with_structured_output(HelpDeskScore)

judge_prompt = """IT 헬프데스크 응답을 평가하세요.
질문: {q}
응답: {a}
각 항목을 1(매우 나쁨)~5(매우 좋음)로 채점하세요."""

# ── 5단계: 전체 평가 실행 + 결과 표 ──
print(f"{'질문':<20} {'Prompt':<3} {'정확성':>4} {'안내':>4} {'공감':>4} {'합계':>4} {'규칙(단계/공감)'}")
print("-" * 70)

totals = {"A": [], "B": []}
for i, tc in enumerate(test_cases):
    for label in ["A", "B"]:
        ans = responses[label][i]
        # LLM 평가
        score = judge.invoke(judge_prompt.format(q=tc["question"], a=ans))
        total = score.accuracy + score.step_guide + score.empathy
        totals[label].append(total)
        # 규칙 평가
        rules = rule_eval(ans)
        r_str = f"{'O' if rules['steps'] else 'X'}/{' O' if rules['empathy'] else 'X'}"
        print(f"{tc['question']:<20} {label:<3} {score.accuracy:>4} {score.step_guide:>4} {score.empathy:>4} {total:>4} {r_str:>10}")

# ── 결론 ──
avg_a = sum(totals["A"]) / len(totals["A"])
avg_b = sum(totals["B"]) / len(totals["B"])
print(f"\n평균 합계 — Prompt A: {avg_a:.1f} / Prompt B: {avg_b:.1f}")
winner = "A" if avg_a > avg_b else "B"
print(f"결론: Prompt {winner}이 더 우수 (상세 시스템 프롬프트가 품질을 크게 개선)")

---

### 생각해보기

1. 평가 파이프라인을 CI/CD에 통합하여, 코드가 변경될 때마다 자동으로 평가를 실행한다면 어떤 이점이 있을까요? 어떤 지표가 특정 임계값 이하로 떨어지면 배포를 차단해야 할까요?
2. 시간이 지나면서 사용자의 기대 수준이 높아지거나, 도메인 지식이 변할 수 있습니다. 평가 데이터셋과 기준을 어떤 주기로, 어떻게 업데이트해야 할까요?
3. 이 노트북에서 배운 평가 기법들을 종합하면, 노트북 1~13에서 만든 챗봇의 품질을 어떻게 체계적으로 측정할 수 있을까요? 전체 커리큘럼을 돌아보며 설계해보세요.